# Question 1.3 — What the Paper Claims to Improve

**Paper:** Efficiently Learning the Accuracy of Labeling Sources for Selective Sampling — Donmez, Carbonell, Schneider (KDD 2009)

**Student:** Yashi Gupta (Roll No. 230072)

## Main Baseline: Repeated Labeling (Sheng et al., KDD 2008)

The primary baseline that IEThresh is compared against throughout the paper is the **Repeated Labeling** method proposed by Sheng, Provost, and Ipeirotis in "Get Another Label? Improving Data Quality and Data Mining Using Multiple, Noisy Labelers" (KDD 2008). This method is referred to as "Repeated" in the paper's experimental results (Figures 3, 4, 5). In Repeated Labeling, the approach is straightforward: for every selected instance, ALL available oracles are queried, and the majority vote among all of them is taken as the final label. There is no attempt to distinguish between reliable and unreliable oracles — everyone gets an equal say, every time.

## Limitation of the Baseline

The key limitation the authors identify is that Repeated Labeling treats all labelers as equally valuable. It queries every oracle for every instance, regardless of whether some oracles have demonstrated themselves to be clearly unreliable. This has two concrete consequences:

1. **Wasted labeling budget:** Every query to an oracle has a cost. If you have 10 oracles but only 3 are actually reliable, Repeated Labeling spends 70% of its budget on oracles that contribute noise rather than useful information. Given a fixed total budget, this means fewer instances can be labeled overall.

2. **Degraded label quality:** Including poor labelers in the majority vote can actively hurt the estimated label, especially when the ratio of unreliable to reliable labelers is high. If 6 out of 10 oracles have accuracy around 0.55 and the other 4 have accuracy around 0.90, the majority vote is dominated by the mediocre majority.

As the authors note in Section 1, Sheng et al. make "the strong and often unrealistic simplifying assumption that all labelers are identical, with the same probability of making a labeling mistake." The real world is not like this — oracle quality varies substantially, and ignoring this variation is wasteful at best and harmful at worst.

## How IEThresh Overcomes This Limitation

IEThresh overcomes this by using confidence-interval-based filtering (Equation 4) to progressively identify and exclude unreliable oracles. Rather than querying all oracles uniformly, IEThresh maintains an upper confidence bound on each oracle's estimated accuracy (Equation 1) and only selects those oracles whose bound meets the threshold criterion: UI(a) ≥ ε × max_a UI(a). This focuses the labeling budget on the most reliable oracles and produces cleaner majority votes from a trusted subset. The result, demonstrated in Figures 3 and 4, is that IEThresh achieves better classification accuracy with fewer total oracle queries — it gets more value per unit of labeling cost.

## A Scenario Where IEThresh Would NOT Outperform Repeated Labeling

IEThresh would struggle relative to Repeated Labeling when **all oracles have very similar, moderate accuracy** — for example, if all 10 oracles have accuracy uniformly distributed in the narrow range [0.65, 0.70].

Here is why this scenario is problematic for IEThresh:

**The threshold mechanism becomes useless.** IEThresh's core advantage is its ability to detect and filter out substantially inferior oracles. But when all oracles are roughly equally mediocre, the confidence intervals for all oracles converge to similar values. The threshold test (Equation 4) either includes all oracles — in which case IEThresh behaves identically to Repeated Labeling but with the overhead of computing confidence intervals — or it arbitrarily excludes some oracles that happen to have slightly lower upper bounds due to sampling noise, losing their contributions without any genuine quality-based reason.

**Exploration overhead hurts.** During the early phase, IEThresh needs to explore oracles to estimate their quality. It spends labeling budget trying to find differences in oracle accuracy that don't meaningfully exist. This exploration cost is wasted — the algorithm is searching for a signal (quality variation across oracles) that is essentially absent. In contrast, Repeated Labeling immediately aggregates all oracles from the very first iteration, obtaining the best possible majority vote right away.

**More oracles in majority vote is strictly better here.** When all oracles have similar accuracy above 0.5, the Condorcet jury theorem tells us that including more independent voters improves the majority vote's accuracy. Repeated Labeling, which always includes all oracles, maximizes this effect. IEThresh, by potentially excluding some oracles, can only do equal or worse in terms of per-instance label quality.

This analysis is consistent with the paper's own experimental results. Figures 3 and 4 show that IEThresh's advantage is most pronounced in scenarios where there is a clear separation between a group of high-quality oracles and a group of low-quality oracles (the synthetic oracle configurations described in Section 4). When the oracle quality gap is large, the threshold mechanism efficiently separates the two groups. When there is no meaningful gap, the mechanism has nothing useful to exploit.